In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
import copy

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score
from sklearn.utils import class_weight, compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.nn import Conv1d, LSTM, GRU
from torch.functional import F
from torch.utils.data import Dataset, TensorDataset, DataLoader
import pytorch_lightning
import torchmetrics
import torchsummary
from torch.nn import Sequential
from torch.nn.modules import Module
from torch.optim import Adam, Adamax, Adagrad, AdamW, Adafactor, Adadelta
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, ExponentialLR
#from torch.fft import fft, fftfreq, fftshift

from scipy.fft import fft

import skorch
from skorch import NeuralNetClassifier
from skorch.dataset import ValidSplit
from skorch.callbacks import EarlyStopping, Checkpoint, LRScheduler, EpochScoring, EpochTimer

from collections import defaultdict
import time

f:\Conda\envs\DeepLearning-Torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


# **Load Dataset**

In [3]:
X_windows = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X2_alt.npy")
y_categorical = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y2.npy")

In [6]:
X_windows = torch.FloatTensor(X_windows)
y_categorical = torch.FloatTensor(y_categorical)
y_encoded = y_integer = np.argmax(y_categorical, axis=1)

In [7]:
# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)
X_val, X_test, y_val, y_test = train_test_split(
    X_windows, y_categorical, test_size=0.5, random_state=42, stratify=y_categorical
)
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

Train: torch.Size([3686, 22, 2, 50]), torch.Size([3686, 4])
Test: torch.Size([2304, 22, 2, 50]), torch.Size([2304, 4])


In [8]:
# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [9]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [10]:
# Iterate through training data
for batch_idx, (eeg_data, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"  EEG data shape: {eeg_data.shape}")  # [batch_size, channels, time_points]
    print(f"  Labels shape: {labels.shape}")      # [batch_size]
    
    # Your model training code goes here
    # predictions = model(eeg_data)
    # loss = criterion(predictions, labels)
    
    if batch_idx == 2:  # Show first 3 batches
        break

# Check one batch from each loader
train_batch = next(iter(train_loader))
val_batch = next(iter(valid_loader))

print(f"\nTrain batch: {train_batch[0].shape}, {train_batch[1].shape}")
print(f"Validation batch: {val_batch[0].shape}, {val_batch[1].shape}")

Batch 0:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])
Batch 1:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])
Batch 2:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])

Train batch: torch.Size([32, 22, 2, 50]), torch.Size([32, 4])
Validation batch: torch.Size([32, 22, 2, 50]), torch.Size([32, 4])


In [11]:
X_train.shape

torch.Size([3686, 22, 2, 50])

# **Xception Networks**

In [4]:
class XceptionBock(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()

        self.residual = nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=1, padding='same')

        self.main_path = Sequential(
            nn.ReLU(),
            nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=in_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel),

            nn.ReLU(),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=out_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel)
        )

        #addition of residual and main tensor in forward pass
        
        self.pool = nn.MaxPool1d(2)
    def forward(self, x):
        res = self.residual(x)
        x = self.main_path(x)
        x = x + res
        x = self.pool(x)

        return x

In [5]:
class LayerNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=eps)
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, L, C)
        x = self.norm(x)
        return x.permute(0, 2, 1)  # (B, C, L)

class ConvNeXtBlock1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm1d(dim)
        self.pwconv1 = nn.Conv1d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv1d(4 * dim, dim, kernel_size=1)
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        return x + residual

class EEGXceptionNet_2(nn.Module):
    def __init__(self, input_channel=2, num_classes=4, name="EEGXceptionNet"):
        super().__init__()
        self.name = name
        self.num_channels = 22  # ✅ FIXED: now defined!

        # One branch per EEG channel
        self.branched_input = nn.ModuleList([
            nn.Sequential(
                ConvNeXtBlock1D(dim=input_channel),
                nn.Conv1d(input_channel, 32, kernel_size=1)  # Project to 32
            ) for _ in range(self.num_channels)
        ])

        # Shared backbone
        self.input_block = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=8, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.branch_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(64, 32, kernel_size=k, padding='same', groups=32),
                nn.Conv1d(32, 32, kernel_size=1),
                nn.BatchNorm1d(32),
                nn.ReLU(inplace=True)
            ) for k in [3, 5, 8, 11]
        ])

        self.post_concat = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=1, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.xception1 = XceptionBock(128, 128)
        self.xception2 = XceptionBock(128, 256)

        self.squeeze_and_excitation = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 256 // 8),
            nn.ReLU(),
            nn.Linear(256 // 8, 256),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        self.Soft=nn.Softmax(dim=1)

    def forward(self, x):
        """
        x: (B, 22, 2, 50)
        """
        B, C, F, T = x.shape
        assert C == self.num_channels and F == 2

        # Process each EEG channel independently
        features = []
        for i in range(self.num_channels):
            xi = x[:, i, :, :]  # (B, 2, 50)
            fi = self.branched_input[i](xi)  # (B, 32, ~50)
            features.append(fi)

        x = torch.stack(features, dim=1)  # (B, 22, 32, L)
        x = x.mean(dim=1)  # (B, 32, L)

        x = self.input_block(x) 
        branches = [branch(x) for branch in self.branch_convs]
        x = torch.cat(branches, dim=1)
        x = self.post_concat(x)
        x = self.xception1(x)
        x = self.xception2(x)

        # Squeeze-and-Excitation
        se_weights = self.squeeze_and_excitation(x)  # (B, 256)
        se_weights = se_weights.unsqueeze(-1)  # (B, 256, 1)
        x = x * se_weights

        x = self.classifier(x)
        x = self.Soft(x)
        return x

# **Test Network Flow with dummy tensor**

In [12]:
X = X_train[0:100]
y = y_train[0:100]
X = torch.rand_like(torch.tensor(X), dtype=torch.float32)
X.shape, X.dtype, X[0].shape

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_28292\3367547783.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.rand_like(torch.tensor(X), dtype=torch.float32)


(torch.Size([100, 22, 2, 50]), torch.float32, torch.Size([22, 2, 50]))

In [13]:
model = EEGXceptionNet_2()
out = model(X)
print("output shape: ",out.shape)
print(model)

output shape:  torch.Size([100, 4])
EEGXceptionNet_2(
  (branched_input): ModuleList(
    (0-21): 22 x Sequential(
      (0): ConvNeXtBlock1D(
        (dwconv): Conv1d(2, 2, kernel_size=(7,), stride=(1,), padding=(3,), groups=2)
        (norm): LayerNorm1d(
          (norm): LayerNorm((2,), eps=1e-06, elementwise_affine=True)
        )
        (pwconv1): Conv1d(2, 8, kernel_size=(1,), stride=(1,))
        (act): GELU(approximate='none')
        (pwconv2): Conv1d(8, 2, kernel_size=(1,), stride=(1,))
      )
      (1): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
    )
  )
  (input_block): Sequential(
    (0): Conv1d(32, 64, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (branch_convs): ModuleList(
    (0): Sequential(
      (0): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=same, groups=32)
      (1): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
      (2): BatchNorm1d(32

f:\Conda\envs\DeepLearning-Torch\lib\site-packages\torch\nn\modules\conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv1d(


In [14]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

count_parameters(model=model)

323832

In [15]:
model.parameters

<bound method Module.parameters of EEGXceptionNet_2(
  (branched_input): ModuleList(
    (0-21): 22 x Sequential(
      (0): ConvNeXtBlock1D(
        (dwconv): Conv1d(2, 2, kernel_size=(7,), stride=(1,), padding=(3,), groups=2)
        (norm): LayerNorm1d(
          (norm): LayerNorm((2,), eps=1e-06, elementwise_affine=True)
        )
        (pwconv1): Conv1d(2, 8, kernel_size=(1,), stride=(1,))
        (act): GELU(approximate='none')
        (pwconv2): Conv1d(8, 2, kernel_size=(1,), stride=(1,))
      )
      (1): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
    )
  )
  (input_block): Sequential(
    (0): Conv1d(32, 64, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (branch_convs): ModuleList(
    (0): Sequential(
      (0): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=same, groups=32)
      (1): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
      (2): BatchNorm1d(32,

# **Training Loop**

In [40]:
# ----------------------------
# Helper: Prepare Data Loaders
# ----------------------------
def prepare_data_loaders_for_eegxception2(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test, batch_size=32):
    """
    Handles (B, 22, 2, 50) format directly — no reshape needed!
    Labels: accepts one-hot or integer.
    """
    # Convert to tensors
    X_train = torch.FloatTensor(X_train)
    X_val = torch.FloatTensor(X_val)
    
    # Handle labels
    if y_train.ndim > 1 and y_train.shape[1] > 1:  # one-hot
        y_train = torch.LongTensor(y_train.argmax(axis=1))
        y_val = torch.LongTensor(y_val.argmax(axis=1))
    else:
        y_train = torch.LongTensor(y_train)
        y_val = torch.LongTensor(y_val)

    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    if X_test is not None:
        X_test = torch.FloatTensor(X_test)
        if y_test.ndim > 1:
            y_test = torch.LongTensor(y_test.argmax(axis=1))
        else:
            y_test = torch.LongTensor(y_test)
        test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    else:
        test_loader = None

    return train_loader, val_loader, test_loader


# ----------------------------
# Training Function
# ----------------------------
def train_eegxception2_model(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    num_classes=4,
    model_type='EEGXceptionNet_2_subject_2',
    learning_rate=0.0005,
    batch_size=64,
    epochs=500,
    use_class_weights=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    patience=20,
    min_lr=1e-6,
    factor=0.5,
    lr_patience=8
):
    print(f"Using device: {device}")
    print(f"Input shape: {X_train.shape} → Expected: (B, 22, 2, 50)")

    # Prepare data loaders
    train_loader, val_loader, test_loader = prepare_data_loaders_for_eegxception2(
        X_train, y_train, X_val, y_val, X_test, y_test, batch_size=batch_size
    )

    # Initialize model
    model = EEGXceptionNet_2(input_channel=2, num_classes=num_classes, name=model_type)
    model.to(device)

    if use_class_weights:
        # Ensure y_train_flat is a 1D numpy array of integers
        y_train_flat = y_train if isinstance(y_train, np.ndarray) else y_train.numpy()
        if y_train_flat.ndim > 1:
            y_train_flat = y_train_flat.argmax(axis=1)

        # ✅ For sklearn >= 1.3: BOTH 'classes' and 'y' are required
        unique_classes = np.unique(y_train_flat)
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=unique_classes,
            y=y_train_flat
        )
        
        class_weights = torch.FloatTensor(class_weights).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        print(f"Class weights: {dict(zip(unique_classes, class_weights.cpu().numpy()))}")
    else:
        criterion = nn.CrossEntropyLoss()
    

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=factor, patience=lr_patience, min_lr=min_lr, verbose=True
    )

    # Tracking
    history = defaultdict(list)
    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        start_time = time.time()

        # ---------- Training ----------
        model.train()
        train_loss = 0.0
        train_preds, train_targets = [], []

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)  # softmax applied
            loss = criterion(outputs.log(), y_batch)  

        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_preds_all, train_targets_all = [], []

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)  # raw logits
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = logits.max(1)
            train_total += y_batch.size(0)
            train_correct += predicted.eq(y_batch).sum().item()

            train_preds_all.extend(predicted.cpu().numpy())
            train_targets_all.extend(y_batch.cpu().numpy())
        train_acc = train_correct / train_total
        # End of epoch: compute metrics
        train_acc = accuracy_score(train_targets_all, train_preds_all)
        try:
            train_kappa = cohen_kappa_score(train_targets_all, train_preds_all)
        except:
            train_kappa = 0.0  # in case of only one class predicted

        # ---------- Validation ----------
        model.eval()
        val_preds_all, val_targets_all = [], []
        val_loss = 0.0

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item()
                _, predicted = logits.max(1)
                val_preds_all.extend(predicted.cpu().numpy())
                val_targets_all.extend(y_batch.cpu().numpy())

        val_acc = accuracy_score(val_targets_all, val_preds_all)
        try:
            val_kappa = cohen_kappa_score(val_targets_all, val_preds_all)
        except:
            val_kappa = 0.0

        # Scheduler step (on validation accuracy)
        scheduler.step(val_acc)

        # Log
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.4f}, Kappa: {train_kappa:.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.4f}, Kappa: {val_kappa:.4f} | "
              f"Time: {epoch_time:.2f}s")

        # History
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        history['train_kappa'].append(train_kappa)
        history['val__kappa'].append(val_kappa)

        history['lr'].append(optimizer.param_groups[0]['lr'])

        # Early stopping & model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model.name}.pth')
            print(f"→ New best model saved (val acc: {val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    print(f"\nTraining finished. Best validation accuracy: {best_val_acc:.4f}")

    # Load best model
    model.load_state_dict(torch.load(f'best_{model.name}.pth'))

    # Final evaluation on test set
    test_results = {}
    if test_loader is not None:
        model.eval()
        test_preds, test_targets = [], []
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(device)
                logits = model(x_batch)
                _, predicted = logits.max(1)
                test_preds.extend(predicted.cpu().numpy())
                test_targets.extend(y_batch.numpy())

        test_acc = accuracy_score(test_targets, test_preds)
        print(f"\nTest Accuracy: {test_acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(test_targets, test_preds, target_names=[f"Class {i}" for i in range(num_classes)]))

        test_results = {
            'y_true': test_targets,
            'y_pred': test_preds,
            'accuracy': test_acc
        }

    return model, dict(history), test_results

In [41]:
model, history, test_results = train_eegxception2_model()

Using device: cuda
Input shape: torch.Size([3686, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Class weights: {0: 0.9994577, 1: 1.0005429, 2: 0.9994577, 3: 1.0005429}
Epoch [1/500] | Train Loss: 1.3866, Acc: 0.2558, Kappa: 0.0077 | Val Loss: 1.3842, Acc: 0.2595, Kappa: 0.0127 | Time: 5.23s
→ New best model saved (val acc: 0.2595)
Epoch [2/500] | Train Loss: 1.3763, Acc: 0.3077, Kappa: 0.0769 | Val Loss: 1.3609, Acc: 0.3333, Kappa: 0.1111 | Time: 5.10s
→ New best model saved (val acc: 0.3333)
Epoch [3/500] | Train Loss: 1.3599, Acc: 0.3131, Kappa: 0.0841 | Val Loss: 1.3385, Acc: 0.3585, Kappa: 0.1447 | Time: 5.08s
→ New best model saved (val acc: 0.3585)
Epoch [4/500] | Train Loss: 1.3348, Acc: 0.3483, Kappa: 0.1311 | Val Loss: 1.3960, Acc: 0.2765, Kappa: 0.0353 | Time: 5.45s
Epoch [5/500] | Train Loss: 1.3086, Acc: 0.3961, Kappa: 0.1948 | Val Loss: 1.2729, Acc: 0.4323, Kappa: 0.2431 | Time: 5.33s
→ New best model saved (val acc: 0.4323)
Epoch [6/500] | Train Loss: 1.2540, Acc: 0.4601, Kappa:

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\783833418.py:204: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.n


Test Accuracy: 0.9444

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.96      0.94      0.95       576
     Class 1       0.96      0.92      0.94       576
     Class 2       0.93      0.95      0.94       576
     Class 3       0.93      0.96      0.94       576

    accuracy                           0.94      2304
   macro avg       0.94      0.94      0.94      2304
weighted avg       0.94      0.94      0.94      2304



In [42]:
history = pd.DataFrame(history)
test_results = pd.DataFrame(test_results)

In [43]:
history.to_csv("F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/train_log_sub2.csv")
test_results.to_csv("F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/test_results_2.csv")

In [44]:
history, test_results

(    train_loss  val_loss  train_acc   val_acc  train_kappa  val__kappa  \
 0     1.386579  1.384154   0.255833  0.259549     0.007702    0.012731   
 1     1.376350  1.360915   0.307651  0.333333     0.076853    0.111111   
 2     1.359938  1.338472   0.313077  0.358507     0.084106    0.144676   
 3     1.334824  1.396044   0.348345  0.276476     0.131119    0.035301   
 4     1.308563  1.272896   0.396093  0.432292     0.194781    0.243056   
 ..         ...       ...        ...       ...          ...         ...   
 81    0.746035  0.744112   0.997830  0.999566     0.997106    0.999421   
 82    0.746222  0.744125   0.997558  0.999566     0.996744    0.999421   
 83    0.745370  0.744103   0.998372  0.999566     0.997830    0.999421   
 84    0.745035  0.744105   0.998644  0.999566     0.998191    0.999421   
 85    0.745035  0.744204   0.998644  0.999566     0.998191    0.999421   
 
           lr  
 0   0.000500  
 1   0.000500  
 2   0.000500  
 3   0.000500  
 4   0.000500  
 .

# Subject 1-9 LOOP

In [ ]:
for sub in [1, 2, 3, 5, 6, 7, 8, 9]:
    print(f"Training Subject Number {sub}")
    X_windows = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X{sub}_alt.npy")
    y_categorical = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y{sub}.npy")
    X_windows = torch.FloatTensor(X_windows)
    y_categorical = torch.FloatTensor(y_categorical)
    y_encoded = y_integer = np.argmax(y_categorical, axis=1)
    # Train-test split
    X_train, X_val, y_train, y_val = train_test_split(
        X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_windows, y_categorical, test_size=0.5, random_state=42, stratify=y_categorical
    )
    # ----------------------------
    # Helper: Prepare Data Loaders
    # ----------------------------
    def prepare_data_loaders_for_eegxception2(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test, batch_size=32):
        """
        Handles (B, 22, 2, 50) format directly — no reshape needed!
        Labels: accepts one-hot or integer.
        """
        # Convert to tensors
        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        
        # Handle labels
        if y_train.ndim > 1 and y_train.shape[1] > 1:  # one-hot
            y_train = torch.LongTensor(y_train.argmax(axis=1))
            y_val = torch.LongTensor(y_val.argmax(axis=1))
        else:
            y_train = torch.LongTensor(y_train)
            y_val = torch.LongTensor(y_val)

        train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        if X_test is not None:
            X_test = torch.FloatTensor(X_test)
            if y_test.ndim > 1:
                y_test = torch.LongTensor(y_test.argmax(axis=1))
            else:
                y_test = torch.LongTensor(y_test)
            test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
            test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        else:
            test_loader = None

        return train_loader, val_loader, test_loader


    # ----------------------------
    # Training Function
    # ----------------------------
    def train_eegxception2_model(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        num_classes=4,
        model_type='EEGXceptionNet_2_subject_2',
        learning_rate=0.0005,
        batch_size=64,
        epochs=500,
        use_class_weights=True,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        patience=20,
        min_lr=1e-6,
        factor=0.5,
        lr_patience=8
    ):
        print(f"Using device: {device}")
        print(f"Input shape: {X_train.shape} → Expected: (B, 22, 2, 50)")

        # Prepare data loaders
        train_loader, val_loader, test_loader = prepare_data_loaders_for_eegxception2(
            X_train, y_train, X_val, y_val, X_test, y_test, batch_size=batch_size
        )

        # Initialize model
        model = EEGXceptionNet_2(input_channel=2, num_classes=num_classes, name=model_type)
        model.to(device)
        print(f"Training model {model.name}")

        if use_class_weights:
            # Ensure y_train_flat is a 1D numpy array of integers
            y_train_flat = y_train if isinstance(y_train, np.ndarray) else y_train.numpy()
            if y_train_flat.ndim > 1:
                y_train_flat = y_train_flat.argmax(axis=1)

            # ✅ For sklearn >= 1.3: BOTH 'classes' and 'y' are required
            unique_classes = np.unique(y_train_flat)
            class_weights = compute_class_weight(
                class_weight='balanced',
                classes=unique_classes,
                y=y_train_flat
            )
            
            class_weights = torch.FloatTensor(class_weights).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            
            print(f"Class weights: {dict(zip(unique_classes, class_weights.cpu().numpy()))}")
        else:
            criterion = nn.CrossEntropyLoss()
        

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=factor, patience=lr_patience, min_lr=min_lr, verbose=True
        )

        # Tracking
        history = defaultdict(list)
        best_val_acc = 0.0
        patience_counter = 0

        for epoch in range(epochs):
            start_time = time.time()

            # ---------- Training ----------
            model.train()
            train_loss = 0.0
            train_preds, train_targets = [], []

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(x_batch)  # softmax applied
                loss = criterion(outputs.log(), y_batch)  

            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            train_preds_all, train_targets_all = [], []

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                logits = model(x_batch)  # raw logits
                loss = criterion(logits, y_batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = logits.max(1)
                train_total += y_batch.size(0)
                train_correct += predicted.eq(y_batch).sum().item()

                train_preds_all.extend(predicted.cpu().numpy())
                train_targets_all.extend(y_batch.cpu().numpy())
            train_acc = train_correct / train_total
            # End of epoch: compute metrics
            train_acc = accuracy_score(train_targets_all, train_preds_all)
            try:
                train_kappa = cohen_kappa_score(train_targets_all, train_preds_all)
            except:
                train_kappa = 0.0  # in case of only one class predicted

            # ---------- Validation ----------
            model.eval()
            val_preds_all, val_targets_all = [], []
            val_loss = 0.0

            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    logits = model(x_batch)
                    loss = criterion(logits, y_batch)
                    val_loss += loss.item()
                    _, predicted = logits.max(1)
                    val_preds_all.extend(predicted.cpu().numpy())
                    val_targets_all.extend(y_batch.cpu().numpy())

            val_acc = accuracy_score(val_targets_all, val_preds_all)
            try:
                val_kappa = cohen_kappa_score(val_targets_all, val_preds_all)
            except:
                val_kappa = 0.0

            # Scheduler step (on validation accuracy)
            scheduler.step(val_acc)

            # Log
            epoch_time = time.time() - start_time
            print(f"Epoch [{epoch+1}/{epochs}] | "
                f"Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.4f}, Kappa: {train_kappa:.4f} | "
                f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.4f}, Kappa: {val_kappa:.4f} | "
                f"Time: {epoch_time:.2f}s")

            # History
            history['train_loss'].append(train_loss / len(train_loader))
            history['val_loss'].append(val_loss / len(val_loader))
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)

            history['train_kappa'].append(train_kappa)
            history['val__kappa'].append(val_kappa)

            history['lr'].append(optimizer.param_groups[0]['lr'])

            # Early stopping & model saving
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), f'best_{model.name}.pth')
                print(f"→ New best model saved (val acc: {val_acc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping triggered.")
                    break

        print(f"\nTraining finished. Best validation accuracy: {best_val_acc:.4f}")

        # Load best model
        model.load_state_dict(torch.load(f'best_{model.name}.pth'))

        # Final evaluation on test set
        test_results = {}
        if test_loader is not None:
            model.eval()
            test_preds, test_targets = [], []
            with torch.no_grad():
                for x_batch, y_batch in test_loader:
                    x_batch = x_batch.to(device)
                    logits = model(x_batch)
                    _, predicted = logits.max(1)
                    test_preds.extend(predicted.cpu().numpy())
                    test_targets.extend(y_batch.numpy())

            test_acc = accuracy_score(test_targets, test_preds)
            print(f"\nTest Accuracy: {test_acc:.4f}")
            print("\nClassification Report:")
            print(classification_report(test_targets, test_preds, target_names=[f"Class {i}" for i in range(num_classes)]))

            test_results = {
                'y_true': test_targets,
                'y_pred': test_preds,
                'accuracy': test_acc
            }

        return model, dict(history), test_results
    
    model, history, test_results = train_eegxception2_model(model_type=f'EEGXceptionNet_2_subject_{sub}')
    history = pd.DataFrame(history)
    test_results = pd.DataFrame(test_results)
    history.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/train_log_sub{sub}.csv")
    test_results.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/test_results_{sub}.csv")

Training Subject Number 3
Using device: cuda
Input shape: torch.Size([3686, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_subject_3
Class weights: {0: 0.9994577, 1: 1.0005429, 2: 0.9994577, 3: 1.0005429}
Epoch [1/500] | Train Loss: 1.3862, Acc: 0.2491, Kappa: -0.0013 | Val Loss: 1.3836, Acc: 0.2999, Kappa: 0.0666 | Time: 5.00s
→ New best model saved (val acc: 0.2999)
Epoch [2/500] | Train Loss: 1.3684, Acc: 0.3256, Kappa: 0.1007 | Val Loss: 1.3377, Acc: 0.3611, Kappa: 0.1481 | Time: 4.50s
→ New best model saved (val acc: 0.3611)
Epoch [3/500] | Train Loss: 1.3354, Acc: 0.3649, Kappa: 0.1532 | Val Loss: 1.3708, Acc: 0.3351, Kappa: 0.1134 | Time: 4.87s
Epoch [4/500] | Train Loss: 1.3021, Acc: 0.4145, Kappa: 0.2194 | Val Loss: 1.3050, Acc: 0.4162, Kappa: 0.2216 | Time: 5.10s
→ New best model saved (val acc: 0.4162)
Epoch [5/500] | Train Loss: 1.2618, Acc: 0.4604, Kappa: 0.2805 | Val Loss: 1.2912, Acc: 0.4184, Kappa: 0.2245 | Time: 4.23s
→ New best model saved (val

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\1730814320.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.9470

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.95      0.95      0.95       576
     Class 1       0.94      0.95      0.95       576
     Class 2       0.94      0.95      0.95       576
     Class 3       0.95      0.94      0.95       576

    accuracy                           0.95      2304
   macro avg       0.95      0.95      0.95      2304
weighted avg       0.95      0.95      0.95      2304

Training Subject Number 5
Using device: cuda
Input shape: torch.Size([3686, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_subject_5
Class weights: {0: 0.9994577, 1: 1.0005429, 2: 0.9994577, 3: 1.0005429}
Epoch [1/500] | Train Loss: 1.3765, Acc: 0.2979, Kappa: 0.0638 | Val Loss: 1.3281, Acc: 0.3776, Kappa: 0.1701 | Time: 4.21s
→ New best model saved (val acc: 0.3776)
Epoch [2/500] | Train Loss: 1.2953, Acc: 0.4094, Kappa: 0.2125 | Val Loss: 1.2597, Acc: 0.4666, Kappa: 0.2888 | Time: 4.11s


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\1730814320.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.9575

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.95      0.95      0.95       576
     Class 1       0.96      0.96      0.96       576
     Class 2       0.95      0.98      0.97       576
     Class 3       0.96      0.94      0.95       576

    accuracy                           0.96      2304
   macro avg       0.96      0.96      0.96      2304
weighted avg       0.96      0.96      0.96      2304

Training Subject Number 6
Using device: cuda
Input shape: torch.Size([3686, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_subject_6
Class weights: {0: 0.9994577, 1: 1.0005429, 2: 0.9994577, 3: 1.0005429}
Epoch [1/500] | Train Loss: 1.3874, Acc: 0.2382, Kappa: -0.0158 | Val Loss: 1.3853, Acc: 0.2687, Kappa: 0.0249 | Time: 4.48s
→ New best model saved (val acc: 0.2687)
Epoch [2/500] | Train Loss: 1.3839, Acc: 0.2732, Kappa: 0.0309 | Val Loss: 1.3753, Acc: 0.2969, Kappa: 0.0625 | Time: 4.22s

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\1730814320.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.

... (3 outputs left out)

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\1730814320.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.9236

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.96      0.92      0.94       576
     Class 1       0.90      0.93      0.92       576
     Class 2       0.92      0.94      0.93       576
     Class 3       0.92      0.91      0.91       576

    accuracy                           0.92      2304
   macro avg       0.92      0.92      0.92      2304
weighted avg       0.92      0.92      0.92      2304

Training Subject Number 9
Using device: cuda
Input shape: torch.Size([3686, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_subject_9
Class weights: {0: 0.9994577, 1: 1.0005429, 2: 0.9994577, 3: 1.0005429}
Epoch [1/500] | Train Loss: 1.3844, Acc: 0.2705, Kappa: 0.0274 | Val Loss: 1.3733, Acc: 0.3095, Kappa: 0.0793 | Time: 5.00s
→ New best model saved (val acc: 0.3095)
Epoch [2/500] | Train Loss: 1.3552, Acc: 0.3326, Kappa: 0.1102 | Val Loss: 1.3531, Acc: 0.3355, Kappa: 0.1140 | Time: 4.64s


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_20076\1730814320.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.9219

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.89      0.93      0.91       576
     Class 1       0.93      0.92      0.93       576
     Class 2       0.92      0.94      0.93       576
     Class 3       0.94      0.89      0.92       576

    accuracy                           0.92      2304
   macro avg       0.92      0.92      0.92      2304
weighted avg       0.92      0.92      0.92      2304



# LOSO LOOP

In [6]:
for sub in [1, 2, 3, 5, 6, 7, 8, 9]:
    print(f"Training Subject Number {sub}")
    X_windows = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/XLoso_{sub}_alt.npy")
    y_categorical = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/yLoso_{sub}.npy")
    X_windows = torch.FloatTensor(X_windows)
    y_categorical = torch.FloatTensor(y_categorical)
    y_encoded = y_integer = np.argmax(y_categorical, axis=1)

    X_test = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X{sub}_alt.npy")
    y_test = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y{sub}.npy")
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test)
    y_test_encoded = y_integer = np.argmax(y_test, axis=1)

    # Train-test split
    X_train, X_val, y_train, y_val = train_test_split(
        X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
    )
    # ----------------------------
    # Helper: Prepare Data Loaders
    # ----------------------------
    def prepare_data_loaders_for_eegxception2(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test, batch_size=32):
        """
        Handles (B, 22, 2, 50) format directly — no reshape needed!
        Labels: accepts one-hot or integer.
        """
        # Convert to tensors
        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        
        # Handle labels
        if y_train.ndim > 1 and y_train.shape[1] > 1:  # one-hot
            y_train = torch.LongTensor(y_train.argmax(axis=1))
            y_val = torch.LongTensor(y_val.argmax(axis=1))
        else:
            y_train = torch.LongTensor(y_train)
            y_val = torch.LongTensor(y_val)

        train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        if X_test is not None:
            X_test = torch.FloatTensor(X_test)
            if y_test.ndim > 1:
                y_test = torch.LongTensor(y_test.argmax(axis=1))
            else:
                y_test = torch.LongTensor(y_test)
            test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
            test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        else:
            test_loader = None

        return train_loader, val_loader, test_loader


    # ----------------------------
    # Training Function
    # ----------------------------
    def train_eegxception2_model(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        num_classes=4,
        model_type='EEGXceptionNet_2_subject_2',
        learning_rate=0.0005,
        batch_size=64,
        epochs=500,
        use_class_weights=True,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        patience=20,
        min_lr=1e-6,
        factor=0.5,
        lr_patience=8
    ):
        print(f"Using device: {device}")
        print(f"Input shape: {X_train.shape} → Expected: (B, 22, 2, 50)")

        # Prepare data loaders
        train_loader, val_loader, test_loader = prepare_data_loaders_for_eegxception2(
            X_train, y_train, X_val, y_val, X_test, y_test, batch_size=batch_size
        )

        # Initialize model
        model = EEGXceptionNet_2(input_channel=2, num_classes=num_classes, name=model_type)
        model.to(device)
        print(f"Training model {model.name}")

        if use_class_weights:
            # Ensure y_train_flat is a 1D numpy array of integers
            y_train_flat = y_train if isinstance(y_train, np.ndarray) else y_train.numpy()
            if y_train_flat.ndim > 1:
                y_train_flat = y_train_flat.argmax(axis=1)

            # ✅ For sklearn >= 1.3: BOTH 'classes' and 'y' are required
            unique_classes = np.unique(y_train_flat)
            class_weights = compute_class_weight(
                class_weight='balanced',
                classes=unique_classes,
                y=y_train_flat
            )
            
            class_weights = torch.FloatTensor(class_weights).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            
            print(f"Class weights: {dict(zip(unique_classes, class_weights.cpu().numpy()))}")
        else:
            criterion = nn.CrossEntropyLoss()
        

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=factor, patience=lr_patience, min_lr=min_lr, verbose=True
        )

        # Tracking
        history = defaultdict(list)
        best_val_acc = 0.0
        patience_counter = 0

        for epoch in range(epochs):
            start_time = time.time()

            # ---------- Training ----------
            model.train()
            train_loss = 0.0
            train_preds, train_targets = [], []

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(x_batch)  # softmax applied
                loss = criterion(outputs.log(), y_batch)  

            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            train_preds_all, train_targets_all = [], []

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                logits = model(x_batch)  # raw logits
                loss = criterion(logits, y_batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = logits.max(1)
                train_total += y_batch.size(0)
                train_correct += predicted.eq(y_batch).sum().item()

                train_preds_all.extend(predicted.cpu().numpy())
                train_targets_all.extend(y_batch.cpu().numpy())
            train_acc = train_correct / train_total
            # End of epoch: compute metrics
            train_acc = accuracy_score(train_targets_all, train_preds_all)
            try:
                train_kappa = cohen_kappa_score(train_targets_all, train_preds_all)
            except:
                train_kappa = 0.0  # in case of only one class predicted

            # ---------- Validation ----------
            model.eval()
            val_preds_all, val_targets_all = [], []
            val_loss = 0.0

            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    logits = model(x_batch)
                    loss = criterion(logits, y_batch)
                    val_loss += loss.item()
                    _, predicted = logits.max(1)
                    val_preds_all.extend(predicted.cpu().numpy())
                    val_targets_all.extend(y_batch.cpu().numpy())

            val_acc = accuracy_score(val_targets_all, val_preds_all)
            try:
                val_kappa = cohen_kappa_score(val_targets_all, val_preds_all)
            except:
                val_kappa = 0.0

            # Scheduler step (on validation accuracy)
            scheduler.step(val_acc)

            # Log
            epoch_time = time.time() - start_time
            print(f"Epoch [{epoch+1}/{epochs}] | "
                f"Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.4f}, Kappa: {train_kappa:.4f} | "
                f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.4f}, Kappa: {val_kappa:.4f} | "
                f"Time: {epoch_time:.2f}s")

            # History
            history['train_loss'].append(train_loss / len(train_loader))
            history['val_loss'].append(val_loss / len(val_loader))
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)

            history['train_kappa'].append(train_kappa)
            history['val__kappa'].append(val_kappa)

            history['lr'].append(optimizer.param_groups[0]['lr'])

            # Early stopping & model saving
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), f'best_{model.name}.pth')
                print(f"→ New best model saved (val acc: {val_acc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping triggered.")
                    break

        print(f"\nTraining finished. Best validation accuracy: {best_val_acc:.4f}")

        # Load best model
        model.load_state_dict(torch.load(f'best_{model.name}.pth'))

        # Final evaluation on test set
        test_results = {}
        if test_loader is not None:
            model.eval()
            test_preds, test_targets = [], []
            with torch.no_grad():
                for x_batch, y_batch in test_loader:
                    x_batch = x_batch.to(device)
                    logits = model(x_batch)
                    _, predicted = logits.max(1)
                    test_preds.extend(predicted.cpu().numpy())
                    test_targets.extend(y_batch.numpy())

            test_acc = accuracy_score(test_targets, test_preds)
            print(f"\nTest Accuracy: {test_acc:.4f}")
            print("\nClassification Report:")
            print(classification_report(test_targets, test_preds, target_names=[f"Class {i}" for i in range(num_classes)]))

            test_results = {
                'y_true': test_targets,
                'y_pred': test_preds,
                'accuracy': test_acc
            }

        return model, dict(history), test_results
    
    model, history, test_results = train_eegxception2_model(model_type=f'EEGXceptionNet_2_LOSO_{sub}')
    history = pd.DataFrame(history)
    test_results = pd.DataFrame(test_results)
    history.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/LOSO/train_log_sub{sub}.csv")
    test_results.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Channelwise_Input_IEXNet/LOSO/test_results_{sub}.csv")

Training Subject Number 1
Using device: cuda
Input shape: torch.Size([8051, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_LOSO_1
Class weights: {0: 1.062698, 1: 0.96488494, 2: 0.9704677, 3: 1.0078869}
Epoch [1/500] | Train Loss: 1.3859, Acc: 0.2622, Kappa: 0.0117 | Val Loss: 1.3772, Acc: 0.3140, Kappa: 0.0897 | Time: 9.08s
→ New best model saved (val acc: 0.3140)
Epoch [2/500] | Train Loss: 1.3701, Acc: 0.3154, Kappa: 0.0913 | Val Loss: 1.3642, Acc: 0.3239, Kappa: 0.1007 | Time: 9.59s
→ New best model saved (val acc: 0.3239)
Epoch [3/500] | Train Loss: 1.3510, Acc: 0.3428, Kappa: 0.1264 | Val Loss: 1.3498, Acc: 0.3428, Kappa: 0.1315 | Time: 9.23s
→ New best model saved (val acc: 0.3428)
Epoch [4/500] | Train Loss: 1.3202, Acc: 0.3800, Kappa: 0.1762 | Val Loss: 1.3568, Acc: 0.3358, Kappa: 0.1126 | Time: 9.29s
Epoch [5/500] | Train Loss: 1.2897, Acc: 0.4258, Kappa: 0.2358 | Val Loss: 1.3169, Acc: 0.3944, Kappa: 0.1983 | Time: 8.74s
→ New best model saved (val acc

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_11272\1702739886.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.2598

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.22      0.21      0.22      1152
     Class 1       0.25      0.25      0.25      1152
     Class 2       0.29      0.25      0.27      1152
     Class 3       0.28      0.33      0.30      1152

    accuracy                           0.26      4608
   macro avg       0.26      0.26      0.26      4608
weighted avg       0.26      0.26      0.26      4608

Training Subject Number 2
Using device: cuda
Input shape: torch.Size([8076, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_LOSO_2
Class weights: {0: 1.0878233, 1: 0.9739508, 2: 0.95011765, 3: 0.9985163}
Epoch [1/500] | Train Loss: 1.3850, Acc: 0.2557, Kappa: 0.0180 | Val Loss: 1.3741, Acc: 0.3104, Kappa: 0.0831 | Time: 12.17s
→ New best model saved (val acc: 0.3104)
Epoch [2/500] | Train Loss: 1.3704, Acc: 0.3026, Kappa: 0.0723 | Val Loss: 1.4149, Acc: 0.2906, Kappa: 0.0536 | Time: 12.41s


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_11272\1702739886.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.2559

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.27      0.21      0.23      1152
     Class 1       0.30      0.28      0.29      1152
     Class 2       0.23      0.26      0.25      1152
     Class 3       0.24      0.27      0.25      1152

    accuracy                           0.26      4608
   macro avg       0.26      0.26      0.26      4608
weighted avg       0.26      0.26      0.26      4608

Training Subject Number 3
Using device: cuda
Input shape: torch.Size([8025, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_LOSO_3
Class weights: {0: 1.0378945, 1: 0.9678003, 2: 0.97961426, 3: 1.0178844}
Epoch [1/500] | Train Loss: 1.3816, Acc: 0.2784, Kappa: 0.0369 | Val Loss: 1.3676, Acc: 0.3293, Kappa: 0.1049 | Time: 12.76s
→ New best model saved (val acc: 0.3293)
Epoch [2/500] | Train Loss: 1.3514, Acc: 0.3534, Kappa: 0.1371 | Val Loss: 1.3383, Acc: 0.3632, Kappa: 0.1504 | Time: 12.85s


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_11272\1702739886.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.2635

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.26      0.21      0.23      1152
     Class 1       0.28      0.26      0.27      1152
     Class 2       0.25      0.35      0.30      1152
     Class 3       0.26      0.23      0.24      1152

    accuracy                           0.26      4608
   macro avg       0.26      0.26      0.26      4608
weighted avg       0.26      0.26      0.26      4608

Training Subject Number 5
Using device: cuda
Input shape: torch.Size([8076, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_LOSO_5
Class weights: {0: 1.0380462, 1: 1.0380462, 2: 0.9796215, 3: 0.95011765}
Epoch [1/500] | Train Loss: 1.3856, Acc: 0.2569, Kappa: 0.0132 | Val Loss: 1.3803, Acc: 0.2941, Kappa: 0.0625 | Time: 12.50s
→ New best model saved (val acc: 0.2941)
Epoch [2/500] | Train Loss: 1.3736, Acc: 0.3019, Kappa: 0.0699 | Val Loss: 1.3723, Acc: 0.2916, Kappa: 0.0584 | Time: 12.24s


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_11272\1702739886.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_{model.


Test Accuracy: 0.2541

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.27      0.25      0.26      1152
     Class 1       0.26      0.24      0.25      1152
     Class 2       0.24      0.34      0.28      1152
     Class 3       0.25      0.18      0.21      1152

    accuracy                           0.25      4608
   macro avg       0.26      0.25      0.25      4608
weighted avg       0.26      0.25      0.25      4608

Training Subject Number 6
Using device: cuda
Input shape: torch.Size([7872, 22, 2, 50]) → Expected: (B, 22, 2, 50)
Training model EEGXceptionNet_2_LOSO_6
Class weights: {0: 1.0181066, 1: 1.025, 2: 0.97910446, 3: 0.97959185}
Epoch [1/500] | Train Loss: 1.3861, Acc: 0.2585, Kappa: 0.0064 | Val Loss: 1.3778, Acc: 0.2917, Kappa: 0.0551 | Time: 12.33s
→ New best model saved (val acc: 0.2917)
Epoch [2/500] | Train Loss: 1.3759, Acc: 0.2872, Kappa: 0.0488 | Val Loss: 1.3664, Acc: 0.3059, Kappa: 0.0741 | Time: 12.42s
→ N

KeyboardInterrupt: 

# **Albumentations**